# Model Loading

Model loading utilities for embedding models, causal LMs (transformers, vLLM),
and cloud API wrappers. All heavy imports (torch, transformers, vllm) are lazy.

In [ ]:
#|default_exp models

In [ ]:
#|export
import os
from dataclasses import dataclass, field
from typing import Any

## Environment and dtype helpers

In [ ]:
#|export
def set_model_env(hf_cache_dir: str | None = None, *, offline: bool = True) -> None:
    """Set environment variables for HuggingFace model loading.

    Args:
        hf_cache_dir: Override HF cache directory. If None, uses HF_HOME if set.
        offline: If True (default), set HF_HUB_OFFLINE=1 for compute nodes with
            no internet. Set to False for local/API usage where models may need
            to be downloaded.
    """
    if offline:
        os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:512")
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
    os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
    if hf_cache_dir is not None:
        os.environ["HF_HUB_CACHE"] = hf_cache_dir

In [ ]:
#|export
def _resolve_dtype(dtype: str) -> Any:
    """Resolve a string dtype to a torch dtype. Lazy torch import."""
    import torch
    mapping = {
        "float16": torch.float16,
        "bfloat16": torch.bfloat16,
        "float32": torch.float32,
    }
    if dtype not in mapping:
        raise ValueError(f"Unknown dtype {dtype!r}, expected one of {list(mapping)}")
    return mapping[dtype]

## Embedding model

In [ ]:
#|export
@dataclass
class EmbeddingModel:
    """Wrapper around a SentenceTransformer model with a convenience encode method."""
    model: Any
    model_name: str
    device: str
    dtype: str

    def encode(self, texts: list[str], batch_size: int = 64, **kwargs) -> Any:
        """Encode texts into embeddings.

        Args:
            texts: List of strings to encode.
            batch_size: Batch size for encoding.
            **kwargs: Additional arguments passed to SentenceTransformer.encode().

        Returns:
            numpy array of shape (len(texts), embedding_dim).
        """
        return self.model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=kwargs.pop("show_progress_bar", len(texts) > batch_size),
            **kwargs,
        )

In [ ]:
#|export
def load_embedding_model(model_name: str = "BAAI/bge-large-en-v1.5", *,
                         device: str = "cuda",
                         dtype: str = "float16",
                         offline: bool = True,
                         st_kwargs: dict | None = None) -> EmbeddingModel:
    """Load a SentenceTransformer embedding model.

    Args:
        model_name: HuggingFace model ID.
        device: Device to load onto ("cuda", "cpu").
        dtype: Model precision ("float16", "bfloat16", "float32").
        offline: If True (default), force offline mode (compute nodes). False allows downloads.
        st_kwargs: Extra kwargs passed directly to the SentenceTransformer constructor
            (e.g. trust_remote_code). Overrides defaults.

    Returns:
        EmbeddingModel wrapping the loaded SentenceTransformer.
    """
    set_model_env(offline=offline)
    import torch
    from sentence_transformers import SentenceTransformer

    torch_dtype = _resolve_dtype(dtype)
    model = SentenceTransformer(
        model_name,
        device=device,
        model_kwargs={"torch_dtype": torch_dtype},
        **(st_kwargs or {}),
    )
    return EmbeddingModel(model=model, model_name=model_name, device=device, dtype=dtype)

## Causal LM (transformers backend)

In [ ]:
#|export
@dataclass
class LLM:
    """Wrapper around a causal LM with tokenizer and a convenience generate method."""
    model: Any
    tokenizer: Any
    model_name: str
    device: str
    dtype: str

    def generate(self, prompts: list[str] | str, *,
                 max_new_tokens: int = 60,
                 system_message: str | None = None,
                 json_schema: dict | None = None,
                 **kwargs) -> list[str]:
        """Generate text completions for one or more prompts.

        Applies the chat template if the tokenizer has one and system_message
        is provided or the tokenizer expects chat format.

        Args:
            prompts: Single prompt string or list of prompts.
            max_new_tokens: Maximum tokens to generate per prompt.
            system_message: Optional system message prepended to each prompt.
            json_schema: Optional JSON schema dict to constrain output.
                Requires the ``outlines`` package.
            **kwargs: Additional arguments passed to model.generate().

        Returns:
            List of generated response strings (new tokens only).
        """
        import torch

        if isinstance(prompts, str):
            prompts = [prompts]

        # Build chat messages and apply template
        all_texts = []
        for prompt in prompts:
            messages = []
            if system_message:
                messages.append({"role": "system", "content": system_message})
            messages.append({"role": "user", "content": prompt})
            text = self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            all_texts.append(text)

        # Use outlines Generator for structured JSON output
        if json_schema is not None:
            try:
                import outlines
            except ImportError:
                raise ImportError(
                    "The `outlines` package is required for structured JSON output "
                    "with the transformers backend. Install it with: "
                    "pip install outlines"
                )
            outlines_model = outlines.from_transformers(self.model, self.tokenizer)
            generator = outlines.Generator(outlines_model, outlines.json_schema(json_schema))
            return generator.batch(all_texts, max_new_tokens=max_new_tokens)

        # Tokenize with left-padding
        inputs = self.tokenizer(
            all_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
        ).to(self.model.device)

        input_len = inputs["input_ids"].shape[1]

        temperature = kwargs.pop("temperature", 0.0)
        top_p = kwargs.pop("top_p", 1.0)
        top_k_val = kwargs.pop("top_k", -1)
        do_sample = kwargs.pop("do_sample", temperature > 0)

        generate_kwargs = dict(max_new_tokens=max_new_tokens, do_sample=do_sample, **kwargs)
        if do_sample:
            generate_kwargs["temperature"] = temperature
            if top_p < 1.0:
                generate_kwargs["top_p"] = top_p
            if top_k_val > 0:
                generate_kwargs["top_k"] = top_k_val

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                **generate_kwargs,
            )

        # Decode only new tokens
        responses = []
        for output in outputs:
            new_tokens = output[input_len:]
            text = self.tokenizer.decode(new_tokens, skip_special_tokens=True)
            responses.append(text.strip())

        return responses

## Causal LM (vLLM backend)

In [ ]:
#|export
@dataclass
class VllmLLM:
    """Wrapper around vLLM's offline LLM engine with the same generate() interface as LLM."""
    engine: Any
    model_name: str
    device: str
    dtype: str

    def generate(self, prompts: list[str] | str, *,
                 max_new_tokens: int = 60,
                 system_message: str | None = None,
                 json_schema: dict | None = None,
                 **kwargs) -> list[str]:
        """Generate text completions using vLLM continuous batching.

        Args:
            prompts: Single prompt string or list of prompts.
            max_new_tokens: Maximum tokens to generate per prompt.
            system_message: Optional system message prepended to each prompt.
            json_schema: Optional JSON schema dict to constrain output.
            **kwargs: Additional arguments (temperature extracted, rest ignored).

        Returns:
            List of generated response strings.
        """
        from vllm import SamplingParams

        if isinstance(prompts, str):
            prompts = [prompts]

        structured_kwargs = {}
        if json_schema is not None:
            from vllm.sampling_params import StructuredOutputsParams
            structured_kwargs["structured_outputs"] = StructuredOutputsParams(json=json_schema)

        sampling_params = SamplingParams(
            temperature=kwargs.pop("temperature", 0.0),
            top_p=kwargs.pop("top_p", 1.0),
            top_k=kwargs.pop("top_k", -1),
            max_tokens=max_new_tokens,
            **structured_kwargs,
        )

        conversations = []
        for prompt in prompts:
            messages = []
            if system_message:
                messages.append({"role": "system", "content": system_message})
            messages.append({"role": "user", "content": prompt})
            conversations.append(messages)

        outputs = self.engine.chat(conversations, sampling_params, use_tqdm=False)
        return [out.outputs[0].text.strip() for out in outputs]

## Causal LM (API backend)

In [ ]:
#|export
@dataclass
class ApiLLM:
    """Wrapper around adulib.llm for cloud API calls, same generate() interface."""
    model_name: str
    max_concurrent: int = 128

    def generate(self, prompts: list[str] | str, *,
                 max_new_tokens: int = 60,
                 system_message: str | None = None,
                 json_schema: dict | None = None,
                 **kwargs) -> list[str]:
        """Generate text completions via cloud API.

        Uses adulib.llm.async_single with batch_executor for concurrency.

        Args:
            prompts: Single prompt string or list of prompts.
            max_new_tokens: Maximum tokens to generate per prompt.
            system_message: Optional system message.
            json_schema: Optional JSON schema dict to constrain output.
                Builds an OpenAI-compatible ``response_format`` parameter.
            **kwargs: Additional arguments forwarded to adulib async_single.

        Returns:
            List of generated response strings.
        """
        import asyncio
        from adulib.llm import async_single
        from adulib.asynchronous import batch_executor

        if isinstance(prompts, str):
            prompts = [prompts]

        extra_kwargs = dict(kwargs)
        # top_p=1.0 is the default (no restriction); drop it to avoid
        # Anthropic API errors (rejects temperature + top_p together).
        if extra_kwargs.get("top_p", 1.0) == 1.0:
            extra_kwargs.pop("top_p", None)
        # top_k is not standard in the OpenAI API; drop if disabled (-1).
        if extra_kwargs.get("top_k", -1) == -1:
            extra_kwargs.pop("top_k", None)
        if json_schema is not None:
            extra_kwargs["response_format"] = {
                "type": "json_schema",
                "json_schema": {"name": "response", "schema": json_schema},
            }

        async def _call(prompt: str) -> str:
            response_text, _cache_hit, _call_log = await async_single(
                prompt,
                model=self.model_name,
                system=system_message,
                max_tokens=max_new_tokens,
                **extra_kwargs,
            )
            return response_text

        async def _run_all():
            results = await batch_executor(
                _call,
                batch_args=[(p,) for p in prompts],
                concurrency_limit=self.max_concurrent,
            )
            return results

        try:
            asyncio.get_running_loop()
            import nest_asyncio
            nest_asyncio.apply()
        except RuntimeError:
            pass
        return asyncio.run(_run_all())

## Model loaders

In [ ]:
#|export
def _load_vllm(model_name: str, *, device: str = "cuda", dtype: str = "float16",
               tensor_parallel_size: int = 1,
               vllm_kwargs: dict | None = None) -> VllmLLM:
    """Load a causal LM via vLLM's offline engine.

    Args:
        model_name: HuggingFace model ID.
        device: Device (only "cuda" supported by vLLM).
        dtype: Model precision ("float16", "bfloat16", "float32", "fp8").
        tensor_parallel_size: Number of GPUs for tensor parallelism (default 1).
        vllm_kwargs: Extra kwargs passed directly to vLLM's LLM constructor
            (e.g. trust_remote_code, tokenizer_mode). Overrides defaults.

    Returns:
        VllmLLM wrapping the vLLM engine.
    """
    set_model_env()
    from vllm import LLM as _VllmEngine

    kw = dict(
        model=model_name,
        gpu_memory_utilization=0.9,
        tensor_parallel_size=tensor_parallel_size,
        enforce_eager=True,   # safer on aarch64/GH200
        max_model_len=16384,  # enriched O*NET prompts can reach ~8K tokens
    )

    if dtype == "fp8":
        # FP8 quantization: use bfloat16 compute dtype with FP8 weight quantization
        kw["dtype"] = "bfloat16"
        kw["quantization"] = "fp8"
    else:
        kw["dtype"] = "half" if dtype == "float16" else dtype

    if vllm_kwargs:
        kw.update(vllm_kwargs)

    engine = _VllmEngine(**kw)
    return VllmLLM(engine=engine, model_name=model_name, device=device, dtype=dtype)

In [ ]:
#|export
def load_llm(model_name: str = "Qwen/Qwen2.5-7B-Instruct", *,
             device: str = "cuda",
             dtype: str = "float16",
             backend: str = "transformers",
             tensor_parallel_size: int = 1,
             vllm_kwargs: dict | None = None) -> LLM | VllmLLM | ApiLLM:
    """Load a causal language model with the specified backend.

    Args:
        model_name: HuggingFace model ID (or API model name for api backend).
        device: Device to load onto ("cuda", "cpu"). Ignored for api backend.
        dtype: Model precision ("float16", "bfloat16", "float32"). Ignored for api.
        backend: Inference backend, either "transformers" (default), "vllm", or "api".
        tensor_parallel_size: Number of GPUs for tensor parallelism (vllm only, default 1).
        vllm_kwargs: Extra kwargs passed to vLLM's LLM constructor (vllm only).

    Returns:
        LLM, VllmLLM, or ApiLLM wrapping the loaded model.
    """
    if backend == "api":
        return ApiLLM(model_name=model_name)

    if backend == "vllm":
        return _load_vllm(model_name, device=device, dtype=dtype,
                          tensor_parallel_size=tensor_parallel_size,
                          vllm_kwargs=vllm_kwargs)

    set_model_env(offline=device != "cpu")
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    torch_dtype = _resolve_dtype(dtype)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch_dtype,
        low_cpu_mem_usage=True,
        device_map=device,
    )
    model.eval()

    return LLM(
        model=model,
        tokenizer=tokenizer,
        model_name=model_name,
        device=device,
        dtype=dtype,
    )

## Cross-encoder reranking model

In [ ]:
#|export
def load_rerank_model(model_name: str = "BAAI/bge-reranker-v2-m3", *,
                      device: str = "cuda",
                      dtype: str = "float16",
                      trust_remote_code: bool = False) -> Any:
    """Load a cross-encoder reranking model via sentence-transformers.

    Args:
        model_name: HuggingFace cross-encoder model ID.
        device: Device to load onto ("cuda", "cpu").
        dtype: Model precision ("float16", "bfloat16", "float32").
        trust_remote_code: If True, allow loading custom model code from HuggingFace.

    Returns:
        sentence_transformers.CrossEncoder instance.
    """
    set_model_env(offline=(device != "cpu"))
    import torch
    from sentence_transformers import CrossEncoder

    torch_dtype = _resolve_dtype(dtype)
    model = CrossEncoder(
        model_name,
        device=device,
        trust_remote_code=trust_remote_code,
        automodel_args={"torch_dtype": torch_dtype},
    )
    return model